# Redrob Candidate Ranking Pipeline (Latest Version)
This notebook contains the latest offline ranking pipeline with updated plausibility filters.

In [ ]:
%%writefile requirements.txt
sentence-transformers>=2.2.0
numpy>=1.24.0
nbformat>=5.9.0


In [ ]:
%%writefile check_tech.py
import json

filepath = r"c:\Users\rocky\OneDrive\Documents\Hackathon_RedrobAI\[PUB] India_runs_data_and_ai_challenge\India_runs_data_and_ai_challenge\candidates.jsonl"

count = 0
with open(filepath, 'r', encoding='utf-8') as f:
    for line in f:
        c = json.loads(line)
        for s in c.get('skills', []):
            name = s.get('name', '').lower()
            dur = s.get('duration_months', 0)
            if (name == 'qlora' and dur > 40) or (name == 'langchain' and dur > 60) or (name == 'chatgpt' and dur > 48):
                print(f"{c['candidate_id']} - {name}: {dur} months")
                count += 1
                break

print(f"Total candidates claiming impossible modern tech timelines: {count}")


In [ ]:
%%writefile data_loader.py
import json
import logging

logger = logging.getLogger(__name__)

def load_candidates_stream(filepath):
    """
    Generator that stream-loads candidates from the JSONL file.
    """
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    try:
                        candidate = json.loads(line)
                        yield candidate
                    except json.JSONDecodeError as e:
                        logger.error('Failed to decode line in candidates.jsonl', extra={"error": str(e)})
                        continue
    except FileNotFoundError:
        logger.error('candidates.jsonl not found', extra={"filepath": filepath})
        raise

if __name__ == "__main__":
    logging.basicConfig(level=logging.INFO)
    count = 0
    import sys
    if len(sys.argv) > 1:
        filepath = sys.argv[1]
    else:
        filepath = 'candidates.jsonl'
        
    for _ in load_candidates_stream(filepath):
        count += 1
    print(f"Loaded {count} candidates.")


In [ ]:
%%writefile download_model.py
import os
from sentence_transformers import SentenceTransformer

def download_model():
    print("Downloading all-MiniLM-L6-v2 model for offline use...")
    model_name = "all-MiniLM-L6-v2"
    local_dir = os.path.join(os.path.dirname(__file__), "local_model", model_name)
    
    # Download and cache the model locally
    model = SentenceTransformer(model_name)
    model.save(local_dir)
    print(f"Model successfully saved to {local_dir}")
    print("You can now safely run main.py completely offline.")

if __name__ == "__main__":
    download_model()


In [ ]:
%%writefile embedding_engine.py
import os
import numpy as np
from sentence_transformers import SentenceTransformer

class EmbeddingEngine:
    def __init__(self, model_path=None):
        if model_path is None:
            local_path = os.path.join(os.path.dirname(__file__), "local_model", "all-MiniLM-L6-v2")
            if os.path.isdir(local_path):
                model_path = local_path
            else:
                # Fallback: download from HuggingFace (for Colab / first run)
                model_path = "all-MiniLM-L6-v2"
            
        self.model = SentenceTransformer(model_path)
        
        # Define JD Facets text
        self.facets_text = {
            "core_ml": (
                "Production experience with embeddings-based retrieval systems, sentence-transformers, search systems, "
                "ranking evaluation like NDCG, MRR, MAP. Pre-LLM ML production experience in NLP and IR. "
                "Strong scrappy product-engineering attitude, able to ship end-to-end working rankers."
            ),
            "engineering_infra": (
                "Production experience with vector databases or hybrid search infrastructure such as Pinecone, "
                "Weaviate, Qdrant, Milvus, OpenSearch, FAISS. Strong Python skills and code quality. "
                "Experience dealing with embedding drift, index refresh, and large-scale deployment."
            ),
            "nice_to_haves": (
                "LLM fine-tuning experience like LoRA, QLoRA, PEFT. Learning-to-rank models, XGBoost, neural rankers. "
                "Prior exposure to HR-tech, recruiting tech, or marketplace products. "
                "Distributed systems, large-scale inference optimization, and open-source contributions."
            )
        }
        
        # Pre-compute facet embeddings
        facet_texts = [self.facets_text["core_ml"], self.facets_text["engineering_infra"], self.facets_text["nice_to_haves"]]
        facet_embeddings = self.model.encode(facet_texts, normalize_embeddings=True)
        self.facet_vectors = {
            "core_ml": facet_embeddings[0],
            "engineering_infra": facet_embeddings[1],
            "nice_to_haves": facet_embeddings[2]
        }
        
    def batch_encode(self, texts, batch_size=256):
        """
        Encodes a list of texts into normalized embeddings.
        """
        # normalize_embeddings=True allows dot product instead of full cosine similarity
        return self.model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False)
        
    def compute_similarity(self, candidate_embeddings):
        """
        Computes weighted similarity for a batch of candidate embeddings against JD facets.
        Returns array of similarity scores.
        """
        # Weights: core_ml: 45%, engineering_infra: 45%, nice_to_haves: 10%
        # Or more simply, w1=0.45, w2=0.45, w3=0.1
        w_core = 0.45
        w_eng = 0.45
        w_nice = 0.10
        
        # candidate_embeddings is (N, dim)
        # dot product with each facet vector (dim,) -> (N,)
        score_core = np.dot(candidate_embeddings, self.facet_vectors["core_ml"])
        score_eng = np.dot(candidate_embeddings, self.facet_vectors["engineering_infra"])
        score_nice = np.dot(candidate_embeddings, self.facet_vectors["nice_to_haves"])
        
        return (w_core * score_core) + (w_eng * score_eng) + (w_nice * score_nice)


In [ ]:
%%writefile plausibility_filter.py
import datetime

CURRENT_YEAR = 2026

def is_honeypot(candidate):
    """
    Returns True if the candidate exhibits impossible patterns (honeypot), else False.
    """
    profile = candidate.get("profile", {})
    yoe = profile.get("years_of_experience", 0)
    
    # Check 1: Expert proficiency in many skills with 0 months duration
    skills = candidate.get("skills", [])
    expert_zero_months = 0
    max_skill_duration = 0
    for s in skills:
        if s.get("proficiency") == "expert" and s.get("duration_months", 0) == 0:
            expert_zero_months += 1
        if s.get("duration_months", 0) > max_skill_duration:
            max_skill_duration = s.get("duration_months", 0)
            
    if expert_zero_months >= 5:
        return True
        
    career_history = candidate.get("career_history", [])
    total_career_months = sum(job.get("duration_months", 0) for job in career_history)
    
    # Check 2: Skill duration greater than total career duration by a large margin (allow some overlap or pre-career learning, but not 10 years extra)
    # e.g., skill duration = 120 months (10 years), but career history sum = 24 months.
    if max_skill_duration > (total_career_months + 48): # 4 years padding for college
        return True
        
    # Check 3: Years of experience claimed is way larger than sum of career history + padding
    # yoe is in years, total_career_months is in months
    if (yoe * 12) > (total_career_months + 60): # allow 5 years missing from history
        return True
        
    # Check 4: Education vs Experience paradox and Education Timeline Weirdness
    education = candidate.get("education", [])
    if education:
        earliest_start = min((edu.get("start_year", CURRENT_YEAR) for edu in education), default=CURRENT_YEAR)
        max_end = max((edu.get("end_year", 0) for edu in education), default=0)
        
        # Assuming typical college start is 18, so candidate age is roughly (CURRENT_YEAR - earliest_start) + 18
        # If experience > age - 15 (started working at 15) -> impossible
        estimated_age = (CURRENT_YEAR - earliest_start) + 18
        if yoe > (estimated_age - 15):
            return True

        # Education Weirdness: Bachelors after Masters
        # We can loosely check if a lower degree starts after a higher degree ends
        bachelors_start = 2050
        masters_start = 0
        for edu in education:
            deg = edu.get("degree", "").lower()
            if deg in ["b.e.", "b.tech", "b.sc", "bachelors"]:
                bachelors_start = min(bachelors_start, edu.get("start_year", 2050))
            if deg in ["m.e.", "m.tech", "m.sc", "masters"]:
                masters_start = max(masters_start, edu.get("start_year", 0))

        if masters_start > 0 and bachelors_start != 2050:
            if masters_start < bachelors_start:
                return True # Started Masters before Bachelors!

        # Massive Gap: If they finished education 10+ years before their first job but have low YoE
        if career_history and max_end > 0:
            first_job_start = 2050
            for job in career_history:
                st = job.get("start_date")
                if st:
                    try:
                        yr = int(st.split("-")[0])
                        first_job_start = min(first_job_start, yr)
                    except:
                        pass
            
            if first_job_start != 2050:
                gap = first_job_start - max_end
                if gap > 10 and yoe < 6:
                    # e.g., graduated 2009, started working 2022 (gap 13), but only 4.2 YoE
                    return True
            
    # Check 5: Overlapping dates producing excessive duration
    # Simple check: max end date - min start date across career history
    # If this chronological span is much smaller than sum of durations, they are claiming multiple full-time roles simultaneously
    if career_history:
        try:
            min_date = None
            max_date = None
            for job in career_history:
                start_str = job.get("start_date")
                if start_str:
                    s_date = datetime.datetime.strptime(start_str, "%Y-%m-%d")
                    if min_date is None or s_date < min_date:
                        min_date = s_date
                        
                end_str = job.get("end_date")
                is_current = job.get("is_current", False)
                if is_current or not end_str:
                    e_date = datetime.datetime.now()
                else:
                    e_date = datetime.datetime.strptime(end_str, "%Y-%m-%d")
                if max_date is None or e_date > max_date:
                    max_date = e_date
                    
            if min_date and max_date:
                chronological_months = (max_date.year - min_date.year) * 12 + (max_date.month - min_date.month)
                # If they claim 2x more duration than chronological time
                if total_career_months > (chronological_months * 1.5 + 24):
                    return True
        except ValueError:
            pass # ignore date parse errors
            
    # Check 6: "Time-Traveling Tech Liars" (Honeypot keyword stuffers)
    # E.g. claiming 80 months of QLoRA (released 2023) or LangChain (released late 2022)
    # Assuming current year is ~2026, max possible duration is ~36-48 months
    for s in skills:
        name = s.get("name", "").lower()
        dur = s.get("duration_months", 0)
        if name == "qlora" and dur > 36:
            return True
        if name == "langchain" and dur > 48:
            return True
        if name == "chatgpt" and dur > 44:
            return True
        if name == "llama-2" and dur > 36:
            return True

    # Check 7: Skills Without Evidence Trap
    # If they claim advanced/expert in vector DBs but never mention them in career history
    career_text = " ".join([job.get("description", "").lower() for job in career_history])
    summary_text = profile.get("summary", "").lower()
    full_text = career_text + " " + summary_text
    
    vector_dbs = {"qdrant", "milvus", "pinecone", "weaviate", "faiss", "semantic search"}
    claimed_vector_dbs = [s.get("name", "").lower() for s in skills if s.get("name", "").lower() in vector_dbs]
    
    # If they claim multiple vector DBs in skills, but have ZERO mentions of any vector DBs in career_history
    if len(claimed_vector_dbs) >= 2:
        evidence_found = False
        for db in vector_dbs:
            if db in career_text:
                evidence_found = True
                break
        if not evidence_found:
            return True # Keyword stuffer trap!

    # Check 8: Computer Vision Disqualifier
    # JD explicitly rejects primary CV without NLP/IR. If they admit limited NLP experience or transitioning to NLP.
    if "professional experience there is limited" in full_text and "transitioning toward nlp" in full_text:
        return True
    
    # If they explicitly state their primary expertise is CV
    if "most of my project work has been in cv" in full_text:
        return True

    # Check 9: Duplicate Job Descriptions (Dataset Quality Edge Case)
    # Flags candidate if they have copy-pasted the exact same long description for multiple different roles
    if len(career_history) > 1:
        descriptions = [job.get("description", "").strip() for job in career_history]
        valid_descriptions = [d for d in descriptions if len(d) > 40] # Ignore very short generic descriptions
        if len(valid_descriptions) != len(set(valid_descriptions)):
            return True

    return False




In [ ]:
%%writefile profile_builder.py
def build_candidate_text(candidate):
    """
    Converts candidate JSON into a dense, clean text profile optimized for embedding.
    """
    profile = candidate.get("profile", {})
    title = profile.get("current_title", "")
    yoe = profile.get("years_of_experience", 0)
    summary = profile.get("summary", "")
    
    lines = []
    lines.append(f"Title: {title}")
    lines.append(f"Experience: {yoe} years")
    if summary:
        lines.append(f"Summary: {summary}")
        
    career = candidate.get("career_history", [])
    if career:
        lines.append("Career History:")
        for job in career[:4]: # Take top 4 recent roles to avoid context overflow
            job_title = job.get("title", "")
            company = job.get("company", "")
            desc = job.get("description", "")
            lines.append(f"- {job_title} at {company}. {desc}")
            
    skills = candidate.get("skills", [])
    if skills:
        # Filter top skills to avoid keyword dumping
        # Prioritize expert/advanced or high duration
        top_skills = sorted(skills, key=lambda x: x.get("duration_months", 0), reverse=True)[:15]
        skill_strs = [f"{s.get('name')} ({s.get('proficiency')}, {s.get('duration_months', 0)} months)" for s in top_skills]
        lines.append("Top Skills: " + ", ".join(skill_strs))
        
    return "\n".join(lines)


In [ ]:
%%writefile reasoning_generator.py
def generate_reasoning(candidate, rank, score, semantic_score, signal_score):
    """
    Generates a concise, specific, and accurate 2-sentence rationale.
    """
    profile = candidate.get("profile", {})
    title = profile.get("current_title", "Professional")
    yoe = profile.get("years_of_experience", 0)

    # Get JD-relevant skills specifically, not just top-by-duration
    JD_RELEVANT = {
        "faiss", "pinecone", "weaviate", "qdrant", "milvus", "opensearch",
        "elasticsearch", "embeddings", "embedding", "vector search", "rag",
        "ranking", "ndcg", "mrr", "pytorch", "tensorflow", "transformers",
        "bert", "llm", "llms", "hugging face", "huggingface", "lora", "qlora",
        "peft", "nlp", "xgboost", "deep learning", "machine learning",
        "scikit-learn", "sklearn", "sentence-transformers", "sentence transformers",
        "langchain", "llamaindex", "information retrieval", "learning to rank",
        "python", "docker", "kubernetes", "mlops",
    }

    skills = candidate.get("skills", [])
    relevant_skills = []
    other_skills = []
    for s in sorted(skills, key=lambda x: x.get("duration_months", 0), reverse=True):
        name = s.get("name", "")
        if name.lower() in JD_RELEVANT or any(jd in name.lower() for jd in JD_RELEVANT):
            relevant_skills.append(name)
        else:
            other_skills.append(name)

    # Pick top 3 relevant skills; fall back to top skills if none match
    display_skills = relevant_skills[:3] if relevant_skills else other_skills[:3]

    signals = candidate.get("redrob_signals", {})
    notice = signals.get("notice_period_days", 90)
    resp_rate = signals.get("recruiter_response_rate", 1.0)
    gh_score = signals.get("github_activity_score", 0)

    # Sentence 1: Specific background with JD-relevant skills
    skill_text = f"with production experience in {', '.join(display_skills)}" if display_skills else "with a broad technical background"
    sentence1 = f"{yoe}-year {title} {skill_text}."

    # Sentence 2: Specific, data-driven justification
    parts = []

    # Semantic fit
    if semantic_score > 0.65:
        parts.append("strong semantic alignment with the JD's core ML and retrieval requirements")
    elif semantic_score > 0.5:
        parts.append("solid semantic overlap with the JD's technical requirements")
    else:
        parts.append("moderate technical overlap with the JD")

    # Notice period
    if notice <= 30:
        parts.append("available within 30 days")
    elif notice <= 60:
        parts.append(f"{notice}-day notice period")
    else:
        parts.append(f"longer {notice}-day notice period")

    # Location fit
    loc = profile.get("location", "").lower()
    will_relocate = signals.get("willing_to_relocate", False)
    if any(city in loc for city in ["pune", "noida", "delhi", "mumbai", "hyderabad"]):
        parts.append("Pune/Noida-compatible location")
    elif will_relocate:
        parts.append("willing to relocate to Pune/Noida")
    else:
        parts.append("location mismatch (requires relocation but unwilling)")

    # Response rate and Open to Work
    open_to_work = signals.get("open_to_work_flag", True)
    if not open_to_work:
        parts.append("NOT currently open to work")
    elif resp_rate > 0.7:
        parts.append("high recruiter responsiveness")
    elif resp_rate < 0.3:
        parts.append(f"low historical response rate ({int(resp_rate*100)}%)")

    # GitHub
    if gh_score > 70:
        parts.append("active open-source contributor")

    sentence2 = "; ".join(parts) + "."
    sentence2 = sentence2[0].upper() + sentence2[1:]

    return f"{sentence1} {sentence2}"


In [ ]:
%%writefile scorer.py
import datetime
from plausibility_filter import is_honeypot

# ============================================================================
# Title relevance — hard gate against irrelevant job titles
# ============================================================================
STRONG_TITLE_MATCHES = {
    "machine learning", "ml engineer", "ml ", "deep learning", "nlp",
    "natural language", "ai engineer", "ai research", "search engineer",
    "ranking engineer", "recommendation", "data scientist", "applied ml",
    "research engineer", "research scientist", "applied scientist",
}

WEAK_TITLE_MATCHES = {
    "software engineer", "backend engineer", "data engineer", "platform engineer",
    "devops", "full stack", "python developer", "infrastructure",
}

IRRELEVANT_TITLES = {
    "hr ", "human resource", "recruiter", "talent acquisition",
    "accountant", "finance", "sales", "marketing", "business development",
    "civil", "mechanical", "electrical engineer", "procurement",
    "legal", "compliance", "admin", "office manager", "receptionist",
    "content writer", "graphic design", "ui/ux", "teacher", "professor",
    "computer vision", "cv engineer", "robotics", "speech", "vision"
}

# ============================================================================
# JD skill matching — explicit overlap with the job description
# These are the skills from the ACTUAL job_description.md
# ============================================================================
JD_CORE_SKILLS = {
    "sentence-transformers", "sentence transformers", "faiss", "pinecone",
    "weaviate", "qdrant", "milvus", "opensearch", "elasticsearch",
    "embeddings", "embedding", "vector search", "vector database",
    "rag", "retrieval augmented", "retrieval-augmented",
    "ranking", "ndcg", "mrr", "learning to rank", "learning-to-rank",
    "information retrieval",
}

JD_STRONG_SKILLS = {
    "pytorch", "tensorflow", "transformers", "bert", "llm", "llms",
    "hugging face", "huggingface", "lora", "qlora", "peft",
    "nlp", "natural language processing", "xgboost",
    "deep learning", "neural network", "machine learning",
    "scikit-learn", "sklearn",
}

JD_NICE_SKILLS = {
    "python", "docker", "kubernetes", "aws", "mlops", "ci/cd",
    "distributed systems", "microservices", "langchain", "llamaindex",
}


def calculate_title_multiplier(candidate):
    """Returns a multiplier based on how relevant the job title is to the JD."""
    profile = candidate.get("profile", {})
    title = profile.get("current_title", "").lower()

    for t in IRRELEVANT_TITLES:
        if t in title:
            return 0.10

    for t in STRONG_TITLE_MATCHES:
        if t in title:
            return 1.0

    for t in WEAK_TITLE_MATCHES:
        if t in title:
            return 0.70

    return 0.45


SERVICES_FIRMS = {
    "tcs", "tata consultancy services", "infosys", "wipro", "cognizant",
    "accenture", "capgemini", "hcl", "tech mahindra", "ibm", "l&t infotech",
    "lti", "mindtree", "mphasis", "syntel", "genpact", "genpact ai"
}

def calculate_services_penalty(candidate):
    """
    Returns a significant penalty (0.4) if the candidate's ENTIRE career
    history is at pure IT-services/staffing firms.
    """
    career_history = candidate.get("career_history", [])
    if not career_history:
        return 1.0
    
    for job in career_history:
        company = job.get("company", "").lower()
        is_service = False
        for firm in SERVICES_FIRMS:
            if firm in company:
                is_service = True
                break
        
        # If they have even one job not at a services firm, they are safe
        if not is_service:
            return 1.0
            
    # If we get here, EVERY job was at a services firm
    return 0.4


def calculate_skill_match_score(candidate):
    """
    Calculates explicit skill overlap with the JD requirements.
    Returns a score from 0.0 to 1.0.
    """
    skills = candidate.get("skills", [])
    skill_names = set()
    for s in skills:
        name = s.get("name", "").lower()
        skill_names.add(name)
        # Also check career descriptions for skill mentions
    
    # Also scan career history descriptions for skill evidence
    career_text = ""
    for job in candidate.get("career_history", []):
        career_text += " " + job.get("description", "").lower()
    
    core_hits = 0
    strong_hits = 0
    nice_hits = 0

    for sname in skill_names:
        for jd_skill in JD_CORE_SKILLS:
            if jd_skill in sname or sname in jd_skill:
                core_hits += 1
                break
        for jd_skill in JD_STRONG_SKILLS:
            if jd_skill in sname or sname in jd_skill:
                strong_hits += 1
                break
        for jd_skill in JD_NICE_SKILLS:
            if jd_skill in sname or sname in jd_skill:
                nice_hits += 1
                break

    # Also check career descriptions for core skill evidence
    for jd_skill in JD_CORE_SKILLS:
        if jd_skill in career_text:
            core_hits += 0.5

    core_score = min(1.0, core_hits / 3)
    strong_score = min(1.0, strong_hits / 3)
    nice_score = min(1.0, nice_hits / 2)

    return (core_score * 0.50) + (strong_score * 0.35) + (nice_score * 0.15)


def calculate_signal_score(candidate):
    """Calculates the behavioral signal score (max 1.0)."""
    score = 0.0
    signals = candidate.get("redrob_signals", {})
    profile = candidate.get("profile", {})

    last_active_str = signals.get("last_active_date")
    resp_rate = signals.get("recruiter_response_rate", 0.0)
    interview_rate = signals.get("interview_completion_rate", 0.0)

    if last_active_str:
        try:
            last_active = datetime.datetime.strptime(last_active_str, "%Y-%m-%d")
            days_inactive = max(0, (datetime.datetime(2026, 6, 16) - last_active).days)
            if days_inactive <= 30 and resp_rate > 0.5:
                score += 0.33
            elif days_inactive <= 90 and resp_rate > 0.3:
                score += 0.15
        except ValueError:
            pass

    if interview_rate > 0.8:
        score += 0.17

    notice_days = signals.get("notice_period_days", 90)
    if notice_days <= 30:
        score += 0.16

    loc = profile.get("location", "").lower()
    will_relocate = signals.get("willing_to_relocate", False)
    is_compatible = any(city in loc for city in ["pune", "noida", "delhi", "mumbai", "hyderabad"])
    if is_compatible or will_relocate:
        score += 0.17
    else:
        score *= 0.1 # Disqualifying penalty

    gh_score = signals.get("github_activity_score", -1)
    if gh_score > 50:
        score += 0.17

    # If not open to work, massive penalty to signals
    if not signals.get("open_to_work_flag", True):
        score *= 0.1

    return min(1.0, score)


def score_candidate(candidate, semantic_score):
    """
    Final scoring: fuses semantic similarity, explicit skill match,
    title relevance, and behavioral signals.

    Weights:
      50% semantic similarity (AI contextual matching)
      30% explicit skill match (deterministic, reproducible)
      20% behavioral signals (engagement, availability)
    """
    if is_honeypot(candidate):
        return -100.0, -100.0, 0.0

    skill_match = calculate_skill_match_score(candidate)
    signal_score = calculate_signal_score(candidate)
    title_mult = calculate_title_multiplier(candidate)

    raw_score = (0.50 * semantic_score) + (0.30 * skill_match) + (0.20 * signal_score)

    # Title gate
    final_score = raw_score * title_mult

    # Services-only penalty (Fix 1)
    services_mult = calculate_services_penalty(candidate)
    final_score *= services_mult

    # Heavy penalty for dead profiles
    signals = candidate.get("redrob_signals", {})
    last_active_str = signals.get("last_active_date")
    resp_rate = signals.get("recruiter_response_rate", 0.0)

    days_inactive = 180
    if last_active_str:
        try:
            last_active = datetime.datetime.strptime(last_active_str, "%Y-%m-%d")
            days_inactive = max(0, (datetime.datetime(2026, 6, 16) - last_active).days)
        except ValueError:
            days_inactive = 180

    if days_inactive > 180 or resp_rate < 0.1:
        final_score *= 0.5

    return final_score, semantic_score, signal_score


In [ ]:
%%writefile validate_submission.py
#!/usr/bin/env python3
"""
Validate submission CSV per challenge rules (sections 2–3).
Row 1 = header. Rows 2–101 = exactly 100 data rows. CSV only.
"""

import csv
import re
import sys
from pathlib import Path

REQUIRED_HEADER = ["candidate_id", "rank", "score", "reasoning"]
CANDIDATE_ID_PATTERN = re.compile(r"^CAND_[0-9]{7}$")
DATA_ROW_START = 2
EXPECTED_DATA_ROWS = 100


def validate_submission(csv_path):
    errors = []
    path = Path(csv_path)

    if path.suffix.lower() != ".csv":
        errors.append("Filename must use a .csv extension.")
    elif not path.stem:
        errors.append("Filename must be your registered participant ID (e.g. team_xxx.csv).")

    try:
        with open(path, "r", encoding="utf-8", newline="") as f:
            reader = csv.reader(f)

            try:
                header = next(reader)
            except StopIteration:
                errors.append("Row 1 must be the header row; file is empty.")
                return errors

            # Row 1: column names and their order come from this line only
            if header != REQUIRED_HEADER:
                errors.append(
                    "Row 1 (header) must be exactly:\n"
                    f"  {','.join(REQUIRED_HEADER)}\n"
                    f"Found:\n"
                    f"  {','.join(header)}"
                )

            data_rows = []
            for row in reader:
                if any(cell.strip() for cell in row):
                    data_rows.append(row)

    except UnicodeDecodeError:
        errors.append("File must be UTF-8 encoded.")
        return errors
    except OSError as e:
        errors.append(f"Cannot read file: {e}")
        return errors

    n = len(data_rows)
    if n != EXPECTED_DATA_ROWS:
        errors.append(
            f"After the header (row 1), there must be exactly {EXPECTED_DATA_ROWS} "
            f"data rows (rows {DATA_ROW_START}–{DATA_ROW_START + EXPECTED_DATA_ROWS - 1}); "
            f"found {n}."
        )

    seen_ids = set()
    seen_ranks = set()
    by_rank = []

    for i, cells in enumerate(data_rows):
        row_num = DATA_ROW_START + i

        if len(cells) != len(REQUIRED_HEADER):
            errors.append(
                f"Row {row_num}: expected {len(REQUIRED_HEADER)} columns "
                f"({','.join(REQUIRED_HEADER)}), got {len(cells)}."
            )
            continue

        row = dict(zip(REQUIRED_HEADER, cells))
        cid = row["candidate_id"].strip()
        rank_s = row["rank"].strip()
        score_s = row["score"].strip()

        if not cid:
            errors.append(f"Row {row_num}: candidate_id is required.")
        elif not CANDIDATE_ID_PATTERN.match(cid):
            errors.append(
                f"Row {row_num}: candidate_id must be CAND_XXXXXXX (7 digits)."
            )
        elif cid in seen_ids:
            errors.append(f"Row {row_num}: duplicate candidate_id '{cid}'.")
        else:
            seen_ids.add(cid)

        try:
            rank = int(rank_s)
            if str(rank) != rank_s:
                raise ValueError
            if not 1 <= rank <= 100:
                errors.append(f"Row {row_num}: rank must be between 1 and 100.")
            elif rank in seen_ranks:
                errors.append(f"Row {row_num}: duplicate rank {rank}.")
            else:
                seen_ranks.add(rank)
        except ValueError:
            errors.append(f"Row {row_num}: rank must be an integer (1–100).")
            rank = None

        try:
            score = float(score_s)
        except ValueError:
            errors.append(f"Row {row_num}: score must be a float.")
            score = None

        if rank is not None and score is not None and cid:
            by_rank.append((rank, score, cid))

    missing = set(range(1, 101)) - seen_ranks
    if missing:
        errors.append(
            f"Each rank 1–100 must appear exactly once; missing: {sorted(missing)}"
        )

    by_rank.sort(key=lambda x: x[0])

    for i in range(len(by_rank) - 1):
        r1, s1, _ = by_rank[i]
        r2, s2, _ = by_rank[i + 1]
        if s1 < s2:
            errors.append(
                f"score must be non-increasing by rank: "
                f"rank {r1} ({s1}) < rank {r2} ({s2})."
            )

    for i in range(len(by_rank) - 1):
        r1, s1, c1 = by_rank[i]
        r2, s2, c2 = by_rank[i + 1]
        if s1 == s2 and c1 > c2:
            errors.append(
                f"Equal scores at ranks {r1} and {r2}: "
                f"tie-break requires candidate_id ascending "
                f"({c1!r} > {c2!r})."
            )

    return errors


def main():
    if len(sys.argv) != 2:
        print("Usage: python validate_submission.py <participant_id>.csv")
        sys.exit(1)

    errors = validate_submission(sys.argv[1])
    if errors:
        print(f"Validation failed ({len(errors)} issue(s)):\n")
        for e in errors:
            print(f"- {e}")
        sys.exit(1)

    print("Submission is valid.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile main.py
import sys
import os
import time
import argparse
import csv
from data_loader import load_candidates_stream
from plausibility_filter import is_honeypot
from profile_builder import build_candidate_text
from embedding_engine import EmbeddingEngine
from scorer import score_candidate
from reasoning_generator import generate_reasoning

# Disable all network access from sentence-transformers
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

# ML-relevant keywords with weights for cheap pre-scoring
JD_KEYWORDS = {
    "machine learning": 5, "deep learning": 5, "nlp": 4, "natural language": 4,
    "pytorch": 5, "tensorflow": 4, "scikit": 3, "xgboost": 3,
    "embedding": 5, "vector": 4, "faiss": 5, "rag": 5,
    "retrieval": 4, "ranking": 4, "recommendation": 3,
    "neural": 4, "transformer": 5, "bert": 5, "llm": 4,
    "fine-tun": 4, "lora": 5, "peft": 5, "qlora": 5,
    "hugging": 3, "search engine": 3, "information retrieval": 4,
    "data scien": 3, "computer vision": 3,
    "pinecone": 5, "weaviate": 5, "qdrant": 5, "milvus": 5,
    "opensearch": 4, "elasticsearch": 3,
    "sentence-transformers": 5, "cosine similarity": 4,
    "python": 2, "aws": 1, "docker": 1, "mlops": 3,
    "ndcg": 5, "mrr": 5, "map": 3,
}

SHORTLIST_SIZE = 3000


def cheap_keyword_score(text_lower):
    """Fast keyword-weighted score — no AI model involved."""
    score = 0
    for keyword, weight in JD_KEYWORDS.items():
        if keyword in text_lower:
            score += weight
    return score


def main():
    parser = argparse.ArgumentParser(description="Redrob AI Candidate Ranking System")
    parser.add_argument("--candidates", type=str, default="candidates.jsonl", help="Path to candidates.jsonl")
    parser.add_argument("--out", type=str, default="submission.csv", help="Output CSV path")
    args = parser.parse_args()

    start_time = time.time()

    # ---- Stage 1: Cheap keyword scan on ALL 100k candidates ----
    print("Stage 1: Fast keyword scan on all candidates...", flush=True)
    shortlist = []
    total = 0
    honeypots = 0

    for candidate in load_candidates_stream(args.candidates):
        total += 1
        if total % 20000 == 0:
            print(f"  ... scanned {total}", flush=True)

        if is_honeypot(candidate):
            honeypots += 1
            continue

        text_profile = build_candidate_text(candidate)
        text_lower = text_profile.lower()
        kscore = cheap_keyword_score(text_lower)

        if kscore > 0:
            shortlist.append((kscore, candidate, text_profile[:256]))

    print(f"Stage 1 done: {total} scanned, {honeypots} honeypots, {len(shortlist)} have ML keywords", flush=True)

    shortlist.sort(key=lambda x: x[0], reverse=True)
    shortlist = shortlist[:SHORTLIST_SIZE]
    print(f"Shortlisted top {len(shortlist)} for AI embedding", flush=True)

    stage1_time = time.time()
    print(f"Stage 1 took {stage1_time - start_time:.1f}s", flush=True)

    # ---- Stage 2: AI embedding on shortlisted candidates only ----
    print("Stage 2: Loading AI model...", flush=True)
    engine = EmbeddingEngine()
    print("Model loaded. Encoding shortlisted candidates...", flush=True)

    texts = [item[2] for item in shortlist]
    candidates = [item[1] for item in shortlist]

    batch_size = 512
    all_scores = []
    for batch_start in range(0, len(texts), batch_size):
        batch_end = min(batch_start + batch_size, len(texts))
        batch_texts = texts[batch_start:batch_end]
        batch_cands = candidates[batch_start:batch_end]

        embeddings = engine.batch_encode(batch_texts)
        semantic_scores = engine.compute_similarity(embeddings)

        for i, c in enumerate(batch_cands):
            final_score, sem_s, sig_s = score_candidate(c, semantic_scores[i])
            all_scores.append({
                "candidate_id": c["candidate_id"],
                "score": final_score,
                "semantic": sem_s,
                "signal": sig_s,
                "candidate": c
            })

        print(f"  ... embedded {batch_end}/{len(texts)}", flush=True)

    # Sort descending by score
    all_scores.sort(key=lambda x: x["score"], reverse=True)
    top_100 = all_scores[:100]

    # Generate CSV
    print(f"Writing top 100 to {args.out}...", flush=True)
    with open(args.out, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["candidate_id", "rank", "score", "reasoning"])

        for rank, item in enumerate(top_100, start=1):
            reasoning = generate_reasoning(
                item["candidate"], rank, item["score"], item["semantic"], item["signal"]
            )
            writer.writerow([item["candidate_id"], rank, float(item["score"]), reasoning])

    end_time = time.time()
    print(f"Finished in {end_time - start_time:.1f}s (Stage 1: {stage1_time - start_time:.1f}s, Stage 2: {end_time - stage1_time:.1f}s)", flush=True)

if __name__ == "__main__":
    main()


In [ ]:
!pip install -r requirements.txt
!python download_model.py
!python main.py --candidates /content/candidates.jsonl --out /content/submission.csv
